# AI_EXTRACT — JSON Log Entity Extraction

## Use Case Overview

`AI_EXTRACT` pulls structured fields out of unstructured or semi-structured text using an LLM. Unlike `raw_log:field` VARIANT path notation (which only works when you know exact key names), `AI_EXTRACT` can handle inconsistent formats, nested text, and natural-language descriptions.

**SA use case:** Normalise inconsistent JSON logs, error messages, or free-text event data into structured columns for analysis — without writing complex parsing logic or regex.

**Dataset:** 200 synthetic JSON web server logs stored as VARIANT in `web_server_logs`. Each log contains IP address, HTTP method, path, status code, duration, and error message (sometimes null).

**What we'll demonstrate:**
1. Basic VARIANT path extraction (the old way)
2. AI_EXTRACT on the full JSON log (the AI way)
3. Extracting from the `error` field specifically — where the text is unstructured
4. Building a structured error analysis table

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Data Exploration

Preview the raw VARIANT logs as they arrive — this is what you'd see from an event stream.

In [ ]:
raw_df = pd.read_sql("SELECT log_id, raw_log FROM web_server_logs ORDER BY log_id LIMIT 5", conn)
print(f"Total logs: {pd.read_sql('SELECT COUNT(*) FROM web_server_logs', conn).iloc[0,0]}")
raw_df

## Step 3 — Traditional VARIANT Extraction (baseline)

For well-structured JSON, Snowflake's colon path syntax works perfectly.
This is the fast, deterministic approach for known schemas.

In [ ]:
traditional_df = pd.read_sql("""
    SELECT
        log_id,
        raw_log:ip::STRING        AS ip_address,
        raw_log:method::STRING    AS http_method,
        raw_log:path::STRING      AS request_path,
        raw_log:status::INTEGER   AS status_code,
        raw_log:duration_ms::INTEGER AS duration_ms,
        raw_log:error::STRING     AS error_message,
        raw_log:user_id::STRING   AS user_id
    FROM web_server_logs
    ORDER BY log_id
    LIMIT 10
""", conn)
traditional_df

## Step 4 — AI_EXTRACT on the Full Log

`AI_EXTRACT(text, field_descriptions)` takes:
- `text`: the content to parse (a string — cast VARIANT to STRING)
- `field_descriptions`: a list of field names you want extracted

The LLM understands semantics, so it works even when field names differ across sources.
This is powerful when merging logs from different services with inconsistent schemas.

In [ ]:
extracted_df = pd.read_sql("""
    SELECT
        log_id,
        AI_EXTRACT(raw_log::STRING, ['ip_address', 'http_method', 'api_endpoint', 'http_status_code', 'response_time_ms', 'error_description', 'user_identifier']) AS extracted
    FROM web_server_logs
    WHERE raw_log:error IS NOT NULL
      AND raw_log:error::STRING != 'null'
    ORDER BY log_id
    LIMIT 10
""", conn)
extracted_df

## Step 5 — Parse Extracted Fields and Build Error Analysis

The extracted VARIANT can be flattened into columns. Here we build a structured error analysis table.

In [ ]:
error_analysis_df = pd.read_sql("""
    WITH extracted AS (
        SELECT
            log_id,
            raw_log:status::INTEGER AS status_code,
            AI_EXTRACT(
                raw_log:error::STRING,
                ['error_type', 'affected_service', 'is_transient']
            ) AS error_details
        FROM web_server_logs
        WHERE raw_log:error IS NOT NULL
          AND raw_log:error::STRING != 'null'
    )
    SELECT
        log_id,
        status_code,
        error_details:error_type::STRING       AS error_type,
        error_details:affected_service::STRING AS affected_service,
        error_details:is_transient::STRING     AS is_transient
    FROM extracted
    ORDER BY log_id
""", conn)
print(f"Error logs: {len(error_analysis_df)}")
error_analysis_df

## Step 6 — Interpretation & SA Tips

**When to use AI_EXTRACT vs VARIANT path syntax:**

| Situation | Use |
|---|---|
| Known, consistent JSON schema | `raw_log:field::STRING` (faster, cheaper) |
| Inconsistent field names across sources | `AI_EXTRACT` |
| Unstructured error message text | `AI_EXTRACT` |
| Merging logs from different services | `AI_EXTRACT` |

**SA tips:**
- AI_EXTRACT is ideal for the **schema discovery** phase when ingesting new data sources
- Use it in a one-time migration job to normalise legacy semi-structured data
- The `field_descriptions` list supports natural language descriptions: `['customer email address', 'order total in USD']`
- Store results in a materialised table for repeated queries to avoid re-running the LLM